# Extraction quality — Phase 1a measurement

Interactive companion to `tests/test_extraction_quality.py`. Runs `GLiNERExtractor`
over the 20 hand-labeled conversations in `data/sample_conversations.jsonl` and prints
per-conversation entity / topic / tone recall plus the means, against the Phase 1a
checkpoint (>70% each).

**Requires GLiNER on a GPU.** Both `gliner` and `gliner2` are heavy and load on
construction — run this notebook on the RunPod GPU pod (task #14) or a local GPU box.
On a CPU box the construction cell raises a clear `ImportError`. The recall math below
is identical to the test's `_recall`, so the numbers match
`pytest tests/test_extraction_quality.py -s`.

In [ ]:
import json
import sys
from pathlib import Path

# Resolve the repo root from this notebook's location (notebooks/) and put
# it on sys.path so `from src...` imports work when the kernel starts in
# notebooks/ (where `src` is not otherwise importable).
cwd = Path.cwd()
REPO = cwd.parent if cwd.name == "notebooks" else cwd
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
DATA = REPO / "data" / "sample_conversations.jsonl"

convs = [json.loads(line) for line in DATA.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"Loaded {len(convs)} conversations from {DATA}")

In [ ]:
# Heavy: loads GLiNER-Decoder + GLiNER2. Needs a GPU.
from src.encoding.gliner_extractor import GLiNERExtractor

extractor = GLiNERExtractor()

In [ ]:
def _recall(expected, extracted):
    # Identical to tests/test_extraction_quality.py:_recall.
    if not expected:
        return 1.0
    return len(expected & extracted) / len(expected)

rows = []
entity_recalls, topic_recalls, tone_recalls = [], [], []
for conv in convs:
    full_text = " ".join(f"User: {u} Assistant: {a}" for u, a in conv["turns"])
    result = extractor.extract(full_text)
    er = _recall(set(conv.get("expected_entities", [])), set(result["entities"]))
    tr = _recall(set(conv.get("expected_topics", [])), set(result["topics"]))
    nr = _recall(set(conv.get("expected_tones", [])), set(result["tones"]))
    entity_recalls.append(er); topic_recalls.append(tr); tone_recalls.append(nr)
    rows.append({"id": conv["id"], "entity_recall": er, "topic_recall": tr, "tone_recall": nr})

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df.style.format("{:.2f}", subset=["entity_recall", "topic_recall", "tone_recall"]))
except ImportError:
    for r in rows:
        print(f"{r['id']}: entity={r['entity_recall']:.2f}, topic={r['topic_recall']:.2f}, tone={r['tone_recall']:.2f}")

In [ ]:
mean_entity = sum(entity_recalls) / len(entity_recalls)
mean_topic = sum(topic_recalls) / len(topic_recalls)
mean_tone = sum(tone_recalls) / len(tone_recalls)
print(f"MEAN: entity={mean_entity:.2f}, topic={mean_topic:.2f}, tone={mean_tone:.2f}")
print()
print("Phase 1a checkpoint: entity / topic / tone recall > 0.70 each")
for name, val in [("entity", mean_entity), ("topic", mean_topic), ("tone", mean_tone)]:
    print(f"  {name}: {val:.2f}  {'PASS' if val > 0.70 else 'BELOW TARGET'}")

## Notes

- **This is a measurement, not a gate.** The companion test only asserts recall floors
  > 0.2 (catches a broken extraction path without flaking on model variance). The >0.70
  target here is the Phase 1a quality checkpoint; below it, iterate on the GLiNER2
  schema prompts in `src/encoding/gliner_extractor.py` (`_STABLE_SCHEMA`) and the
  Bonsai relation prompt.
- **Empty expected sets score 1.0** (no expected entities → nothing to miss), matching
  the test's `_recall`.
- **Discovery output** (`discovered` from GLiNER-Decoder) is not scored here; it is
  open-ended and feeds the promotion buffer, not the recall metric.